# Single vs Batch Predict Benchmark (Sweet Spot)

This notebook compares:
- `POST /anonymizer/predict` (single paragraph per request)
- `POST /anonymizer/predict-batch` (multiple paragraphs per request)

It benchmarks latency and throughput and suggests a **sweet spot** batch size.


In [ ]:
from __future__ import annotations

from collections import defaultdict
from statistics import mean, median
from time import perf_counter
import math
import os
from pathlib import Path

import requests
from tqdm import tqdm

from aymurai.experiments.entity_disambiguation.runner import (
    call_extraction_api as extract_document,
)


In [ ]:
# Endpoint and benchmark config
API_URL = "http://localhost:8000"
USE_CACHE = False
TIMEOUT_S = 180

# Real document extraction settings
DATA_ROOT = Path(
    os.getenv(
        "DISAMBIGUATION_DATA_ROOT",
        "../../../resources/data/restricted/disambiguation-eval/files",
    )
)
DOC_EXTENSIONS = {".pdf", ".docx"}
DOC_INDEX = 2

# How many benchmark repetitions per setting
REPEATS = 5

# Candidate client-side batch sizes for /predict-batch
CLIENT_BATCH_SIZES = [1, 2, 4, 8, 16, 32, 64]

# Maximum number of extracted paragraphs used in benchmark
TARGET_TOTAL_PARAGRAPHS = 128


In [ ]:
def discover_documents(root: Path, extensions: set[str]) -> list[Path]:
    extensions = {ext.lower() for ext in extensions}
    return sorted(
        path
        for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() in extensions
    )


documents = discover_documents(DATA_ROOT, DOC_EXTENSIONS)
print(f"Found {len(documents)} documents")
if not documents:
    raise ValueError(f"No documents found in {DATA_ROOT}")

doc_index = min(DOC_INDEX, len(documents) - 1)
doc_path = documents[doc_index]
print(f"Using document: {doc_path}")

session = requests.Session()
document = extract_document(
    session,
    endpoint=f"{API_URL}/misc/document-extract",
    file_path=doc_path,
    timeout_s=300,
)
paragraphs = document["detail"]["document"]
if not paragraphs:
    raise ValueError("Document extraction returned 0 paragraphs")

# Keep only up to TARGET_TOTAL_PARAGRAPHS for stable benchmark runtime
paragraphs = paragraphs[:TARGET_TOTAL_PARAGRAPHS]
print(f"Paragraphs selected for benchmark: {len(paragraphs)}")


In [ ]:
def chunked(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i : i + size]
def call_predict_single(paragraphs: list[str]) -> tuple[float, int]:
    start = perf_counter()
    processed = 0
    for p in paragraphs:
        r = session.post(
            url=f"{API_URL}/anonymizer/predict",
            json={"text": p},
            params={"use_cache": USE_CACHE},
            timeout=TIMEOUT_S,
        )
        r.raise_for_status()
        _ = r.json()
        processed += 1
    elapsed = perf_counter() - start
    return elapsed, processed
def call_predict_batch(paragraphs: list[str], client_batch_size: int) -> tuple[float, int]:
    start = perf_counter()
    processed = 0
    for chunk in chunked(paragraphs, client_batch_size):
        payload = [{"text": p} for p in chunk]
        r = session.post(
            url=f"{API_URL}/anonymizer/predict-batch",
            json=payload,
            params={"use_cache": USE_CACHE},
            timeout=TIMEOUT_S,
        )
        r.raise_for_status()
        data = r.json().get("data", [])
        processed += len(data)
    elapsed = perf_counter() - start
    return elapsed, processed


In [ ]:
# Benchmark runs
rows = []

# 1) Single endpoint benchmark
for run in tqdm(range(1, REPEATS + 1), desc="Single /predict", unit="run"):
    elapsed, processed = call_predict_single(paragraphs)
    rows.append(
        {
            "mode": "single",
            "client_batch_size": 1,
            "run": run,
            "processed": processed,
            "total_s": elapsed,
            "ms_per_paragraph": (elapsed / processed) * 1000 if processed else None,
            "paragraphs_per_s": processed / elapsed if elapsed else None,
        }
    )

# 2) Batch endpoint benchmark
for batch_size in tqdm(CLIENT_BATCH_SIZES, desc="Batch sizes", unit="size"):
    for run in range(1, REPEATS + 1):
        elapsed, processed = call_predict_batch(paragraphs, client_batch_size=batch_size)
        rows.append(
            {
                "mode": "batch",
                "client_batch_size": batch_size,
                "run": run,
                "processed": processed,
                "total_s": elapsed,
                "ms_per_paragraph": (elapsed / processed) * 1000 if processed else None,
                "paragraphs_per_s": processed / elapsed if elapsed else None,
            }
        )

print(f"Collected rows: {len(rows)}")


In [ ]:
by_key = defaultdict(list)
for row in rows:
    key = (row["mode"], row["client_batch_size"])
    by_key[key].append(row)

summary = []
for (mode, bs), group in sorted(by_key.items(), key=lambda x: (x[0][0], x[0][1])):
    total_s_values = [g["total_s"] for g in group]
    mpp_values = [g["ms_per_paragraph"] for g in group if g["ms_per_paragraph"] is not None]
    tps_values = [g["paragraphs_per_s"] for g in group if g["paragraphs_per_s"] is not None]

    summary.append(
        {
            "mode": mode,
            "client_batch_size": bs,
            "runs": len(group),
            "mean_total_s": mean(total_s_values),
            "median_total_s": median(total_s_values),
            "mean_ms_per_paragraph": mean(mpp_values),
            "mean_paragraphs_per_s": mean(tps_values),
        }
    )

header = (
    f"{'mode':<8} {'batch':>6} {'runs':>5} {'mean_total_s':>14} "
    f"{'median_total_s':>15} {'mean_ms/paragraph':>20} {'mean_paragraphs/s':>19}"
)
print(header)
print('-' * len(header))
for s in summary:
    print(
        f"{s['mode']:<8} {s['client_batch_size']:>6} {s['runs']:>5} "
        f"{s['mean_total_s']:>14.3f} {s['median_total_s']:>15.3f} "
        f"{s['mean_ms_per_paragraph']:>20.2f} {s['mean_paragraphs_per_s']:>19.2f}"
    )


In [ ]:
batch_summary = [s for s in summary if s["mode"] == "batch"]
if not batch_summary:
    raise RuntimeError("No batch results found")

best_tps = max(s["mean_paragraphs_per_s"] for s in batch_summary)
threshold = best_tps * 0.95

candidates = [s for s in batch_summary if s["mean_paragraphs_per_s"] >= threshold]
sweet_spot = min(candidates, key=lambda s: s["client_batch_size"])

single = next(s for s in summary if s["mode"] == "single")

print("Best observed batch throughput:", round(best_tps, 2), "paragraphs/s")
print("95% threshold:", round(threshold, 2), "paragraphs/s")
print(
    "Sweet spot client batch size:",
    sweet_spot["client_batch_size"],
    "(mean throughput:",
    round(sweet_spot["mean_paragraphs_per_s"], 2),
    "paragraphs/s)",
)

speedup = sweet_spot["mean_paragraphs_per_s"] / single["mean_paragraphs_per_s"]
print("Speedup vs single:", round(speedup, 2), "x")
